# Timestamp interpolation

GoPro GPMF gives GPS samples but only a coarse `[start, end]` time span per
video frame, not an exact timestamp per sample. This notebook recovers
accurate per-sample timestamps and uses them for lap segmentation and
lap-vs-lap delta.

The recovery is done by `pacer.interpolate_timestamps` (a C++ port of the
original gradient-descent approach); a PyTorch reference is included for
comparison.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

import pacer

# No hard-coded paths: point PACER_DATA at a folder of GoPro .MP4 files,
# or edit the files list directly.
data_dir = Path(os.environ.get("PACER_DATA", "data"))
files = sorted(str(p) for p in data_dir.glob("*.MP4"))
assert files, "No .MP4 files found; set PACER_DATA or edit the files list."
files

## Load GPS samples and their frame spans

Each sample keeps the `[start, end]` time span of the frame it came from. We
drop near-stationary samples (`full_speed <= 3 m/s`) as noise.

In [ ]:
samples, spans = [], []
src = pacer.GPMFSource(files[0])

def on_sample(s, i, n):
    if s.full_speed > 3:
        samples.append(s)
        spans.append(src.current_time_span())

while not src.is_end():
    src.read_samples(on_sample)
    src.next()

len(samples), samples[0]

## Recover per-sample timestamps (C++)

`interpolate_timestamps` fits a constant-frequency model
`t[i] = phase + (cumsum(di)[i] - 1) / frequency` so timestamps stay within
each frame's `[start, end]` span (see `pacer/interpolation`).

In [ ]:
cs = pacer.CoordinateSystem(samples[0])
result = pacer.interpolate_timestamps(samples, spans, cs)
times = result.timestamps
print(f"phase={result.phase:.3f}  frequency={result.frequency:.3f}  "
      f"loss={result.loss:.3e}")

### Reference: the original PyTorch optimizer

Optional sanity check that the C++ result matches the notebook's original
`torch` optimization on the same inputs.

In [ ]:
import torch

rough = len(samples) / len(set(spans))
floor = np.array([b for (b, _) in spans])
ceil = np.array([e for (_, e) in spans])
di = np.concatenate([[1.0], np.round([
    cs.distance(a, b) / (0.5 * a.full_speed + 0.5 * b.full_speed) * rough
    for a, b in zip(samples[:-1], samples[1:])
])])

floor_t = torch.tensor(floor, dtype=torch.float64)
ceil_t = torch.tensor(ceil, dtype=torch.float64)
di_t = torch.tensor(di, dtype=torch.float64)
phase = torch.tensor(float(floor[0]), dtype=torch.float64, requires_grad=True)
frequency = torch.tensor(float(rough), dtype=torch.float64, requires_grad=True)

def make_t():
    return phase + 1.0 / frequency * (di_t.cumsum(0) - 1)

def loss(x):
    nd = (x[1:] - x[:-1]) / di_t[1:]
    spacing = ((nd - nd.mean()) ** 2).mean()
    constraints = ((floor_t - x).clip(min=0) + (x - ceil_t).clip(min=0)) ** 2
    return spacing + constraints.mean()

for lr in [1e-1, 1e-2, 1e-3]:
    opt = torch.optim.Adam([phase, frequency], lr=lr)
    for _ in range(100):
        opt.zero_grad()
        loss(make_t()).backward()
        opt.step()

max_dt = float(np.max(np.abs(make_t().detach().numpy() - np.array(times))))
print(f"torch phase={float(phase):.3f} freq={float(frequency):.3f}; "
      f"max |dt| vs C++ = {max_dt:.2e}")

## Segment into laps

Feed the recovered timestamps into `Laps`, pick a start line and segment.

In [ ]:
laps = pacer.Laps()
laps.set_coordinate_system(cs)
for s, t in zip(samples, times):
    laps.add_point(s, t)

laps.sectors.start_line = laps.pick_random_start()
laps.update()

pd.DataFrame([
    dict(lap=i, lap_time=laps.lap_time(i)) for i in range(laps.laps_count())
])

## Lap-vs-lap time delta

Resample every lap onto a reference lap and plot the cumulative time delta to
the last lap along the track distance.

In [ ]:
reference = laps.get_lap(1)
reference.width = 5
last = laps.laps_count() - 1

rows = []
for i in range(1, last):
    a = reference.resample(laps.get_lap(i), cs)
    b = reference.resample(laps.get_lap(last), cs)
    t0 = np.array([p.time for p in a.points]) - a.points[0].time
    t1 = np.array([p.time for p in b.points]) - b.points[0].time
    n = min(len(a.cum_distances), len(t0), len(t1))
    rows.append(pd.DataFrame(dict(
        distance=np.asarray(a.cum_distances)[:n], delta=(t1 - t0)[:n], lap=i)))

px.line(pd.concat(rows), x="distance", y="delta", color="lap",
        title="Lap time delta vs last lap")